### Creating a training dataset with label as spam or not spam and text

In [27]:
import numpy as np
import re

docs = [
    ('spam', 'win money now'),
    ('spam', 'free money win'),
    ('spam', 'claim your free prize'),
    ('ham',  'meeting schedule tomorrow'),
    ('ham',  'let us schedule a meeting'),
    ('ham',  'project update tomorrow'),
]

### Creating an array of labels marked as 0 or 1

In [28]:
labels = np.array([1 if x == 'spam' else 0 for x, _ in docs])
labels

array([1, 1, 1, 0, 0, 0])

### Creating an array of texts

In [29]:
texts = np.array([x for _, x in docs])
texts

array(['win money now', 'free money win', 'claim your free prize',
       'meeting schedule tomorrow', 'let us schedule a meeting',
       'project update tomorrow'], dtype='<U25')

### Tokenize the text to split a text into array of words

In [30]:
def tokenize(text):
    token =re.findall(r'[a-z]+', text)
    return token
tokenize(texts[0])

['win', 'money', 'now']

### Create a dictionary of all the words found so far in all the texts

In [31]:
vocab = {}
for t in texts:
    for w in tokenize(t):
        if w not in vocab:
            vocab[w] = len(vocab)
print(vocab)

{'win': 0, 'money': 1, 'now': 2, 'free': 3, 'claim': 4, 'your': 5, 'prize': 6, 'meeting': 7, 'schedule': 8, 'tomorrow': 9, 'let': 10, 'us': 11, 'a': 12, 'project': 13, 'update': 14}


### Create an array of the number of occurances of a word in each line of text

In [41]:
X = np.zeros((len(texts), len(vocab)), dtype = int)

def vectorize(texts):
    for i, text in enumerate(texts):
        for w in tokenize(text):
            if w in vocab:
                X[i, vocab[w]] += 1
    return X
print(vectorize(texts))

[[1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 0 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 1 1]]


### Creating prior and liklihood

In [49]:
#aplha is laplace smoothening which is used to handle 0 probs
def fit_nb(X, y, alpha=1.0):
    #converting all labels to integers
    y = y.astype(int)
    #V is the total size of words in X
    V = X.shape[1]
    #creating an array of 0s of size 2 for prob of priors, as prior is either prob of spam or ham
    log_priors = np.zeros(2)
    #creating an array of prob of a word given spam or ham
    log_lik = np.zeros((2, V))
    
    for c in (0, 1):
        #creating a vector for each condition of spam or ham
        Xc = X[y == c]
        # print(Xc)
        #priors is the probability of a doc being spam or ham, so number of rows in Xc / number of rows in X
        log_priors[c] = np.log(Xc.shape[0] / X.shape[0])
        #total count of each word in Xc
        word_counts = Xc.sum(axis=0)
        #print(word_counts)
        smoothed = word_counts + alpha
        #sum of word counts would give total number of words
        denom = smoothed.sum()
        log_lik[c] = np.log(smoothed / denom)

    return log_priors, log_lik

log_priors, log_lik = fit_nb(X, labels, alpha=1.0)
print(log_priors, log_lik)

[-0.69314718 -0.69314718] [[-3.25809654 -3.25809654 -3.25809654 -3.25809654 -3.25809654 -3.25809654
  -3.25809654 -2.15948425 -2.15948425 -2.15948425 -2.56494936 -2.56494936
  -2.56494936 -2.56494936 -2.56494936]
 [-2.12026354 -2.12026354 -2.52572864 -2.12026354 -2.52572864 -2.52572864
  -2.52572864 -3.21887582 -3.21887582 -3.21887582 -3.21887582 -3.21887582
  -3.21887582 -3.21887582 -3.21887582]]


In [50]:
scores = log_priors + X @ log_lik.T
print(scores)

[[-10.46743679  -7.4594029 ]
 [-10.46743679  -7.05393779]
 [-13.72553333 -10.39059665]
 [ -7.17159993 -10.34977466]
 [-12.70696375 -16.7875263 ]
 [ -7.98253014 -10.34977466]]


In [51]:
print(np.argmax(scores, axis = 1))

[1 1 1 0 0 0]


In [55]:
inv_vocab = {i:w for w,i in vocab.items()}
score = log_lik[1] - log_lik[0]
top = np.argsort(-score)[:5]
print([inv_vocab[i] for i in top])

['win', 'money', 'free', 'now', 'claim']
